In [1]:
import pandas as pd

# define source directory 
source_dir = 'tu-06/rnaseq/sample_b/'
# checkthepwdtoensurecorrectpath


# Inpute file
input_file = source_dir + 'input.csv'
trace_file = source_dir + 'trace.csv'

# load the df correctly because the columens are like that
# Index(['name;hash;path;type;size'], dtype='object')
input_df = pd.read_csv(input_file, sep=';')
trace_df = pd.read_csv(trace_file)

In [2]:
# Group by name and show the different files
for name, group in input_df.groupby('name'):
    # print(group[['path', 'size']])
    print(str(group[['name']]) + ' has ' + str(len(group)) + ' files.')
    total_size = group['size'].sum()
    print('Total size: ' + str(total_size) + ' bytes.')

# Assert length of traces is equal to number of unique tasks in input
assert len(trace_df) == len(input_df['name'].unique())


                                                name
4  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI... has 1 files.
Total size: 3147288982 bytes.
                                                name
0  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...
1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom... has 2 files.
Total size: 4063383142 bytes.
                                                 name
9   NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...
10  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_... has 2 files.
Total size: 3985567244 bytes.
                                                 name
17  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...
18  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen... has 2 files.
Total size: 3437932337 bytes.
                                                 name
31  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...
32  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS... has 2 files.
Total size: 175770742 bytes.
                                   

In [3]:
# Make x the sum of input_df grouped by task name
x = input_df.groupby('name')['size'].sum().values
y = trace_df['duration'].values

# Assert lengths are equal
assert len(x) == len(y)

In [7]:
# Make a simple svm, linear regression and random forest regression model to predict y from x
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
X_train, X_test, y_train, y_test = train_test_split(x.reshape(-1, 1), y, test_size=0.1, random_state=42)
models = {
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(),
    'LinearRegression': LinearRegression()
}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f'{name} MSE: {mse}')


SVR MSE: 2374234354.79043
RandomForest MSE: 48398912927.08367
LinearRegression MSE: 2485924163.51955


In [8]:
# Example of how to use the model to predict duration for a new input size
new_input_size = 500000000  # 500 MB
for name, model in models.items():
    predicted_duration = model.predict([[new_input_size]])
    print(f'{name} predicted duration for input size {new_input_size} bytes: {predicted_duration[0]} seconds')


SVR predicted duration for input size 500000000 bytes: 14411.946110872112 seconds
RandomForest predicted duration for input size 500000000 bytes: 21985.805142857145 seconds
LinearRegression predicted duration for input size 500000000 bytes: 77744.10504058322 seconds
